In [1]:
import pickle 
from pathlib import Path 
import yaml
import importlib
import numpy as np 
from itertools import product
import copy


## SSL Configs - single task

In [2]:
## Get list of configs to train 

config_list = sorted(list(Path("../model_configs/").glob("*ssl*.yaml"))) # [1:] since don't need original config
print(config_list)

# config_dict = {ix:str(config) for ix,config in enumerate(config_list)}
# config_dict

[PosixPath('../model_configs/pilot_ssl_audioset_resnet50.yaml'), PosixPath('../model_configs/pilot_ssl_audioset_resnet50_barlow.yaml'), PosixPath('../model_configs/pilot_ssl_audioset_resnet50_simclr.yaml'), PosixPath('../model_configs/pilot_ssl_word_resnet50.yaml'), PosixPath('../model_configs/pilot_ssl_word_resnet50_barlow.yaml'), PosixPath('../model_configs/pilot_ssl_word_resnet50_simclr.yaml')]


In [4]:
## Set hyperparameters to search over 

proj_dim = [np.power(2,pow) for pow in [9, 11, 13]]
print(proj_dim)
batch_size = [256, 768]
print(batch_size)
num_warmup_steps_or_ratio = [0.1] 
lr_scale = [0.6] # scaling factor for lr. lr will be lr_scale * batch_size / 256 

all_combos = list(product(*[proj_dim, proj_dim, batch_size]))
len(all_combos)


[512, 2048, 8192]
[256, 768]


18

In [5]:
## Set up MMCR configs 
config_dir = Path("../model_configs/mmcr_search/")
config_dir.mkdir(parents=True, exist_ok=True)

base_config = yaml.load(open("../model_configs/pilot_ssl_audioset_resnet50.yaml"), Loader=yaml.FullLoader)
base_config

batch_size_256_configs = []
batch_size_768_configs = [] 

for ix, (proj_dim1, proj_dim2, batch_size) in enumerate(all_combos):
    yaml_file_name = config_dir / f"ssl_mmcr_audioset_resnet50_hparam_set_{ix}.yaml"
    config = copy.deepcopy(base_config)
    # update model params 
    config['model']['arch_kwargs']['projector_dims'] = [int(proj_dim1), int(proj_dim2)]

    # update batch_size
    config['hparas']['global_batch_size'] = batch_size

    # set standard params 
    config['lr'] = 0.6 # standard lr scale here 

    if batch_size == 256:
        batch_size_256_configs.append(yaml_file_name.resolve())
    elif batch_size == 768:
        batch_size_768_configs.append(yaml_file_name.resolve())

    with open(yaml_file_name, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

In [6]:
## Set up MMCR configs 
config_dir = Path("../model_configs/mmcr_search/")
config_dir.mkdir(parents=True, exist_ok=True)

base_config = yaml.load(open("../model_configs/pilot_ssl_word_resnet50.yaml"), Loader=yaml.FullLoader)
base_config

for ix, (proj_dim1, proj_dim2, batch_size) in enumerate(all_combos):
    yaml_file_name = config_dir / f"ssl_mmcr_word_resnet50_hparam_set_{ix}.yaml"
    config = copy.deepcopy(base_config)
    # update model params 
    config['model']['arch_kwargs']['projector_dims'] = [int(proj_dim1), int(proj_dim2)]

    # update batch_size
    config['hparas']['global_batch_size'] = batch_size

    # set standard params 
    config['lr'] = 0.6 # standard lr scale here 
    print(yaml_file_name)
    
    if batch_size == 256:
        batch_size_256_configs.append(yaml_file_name.resolve())
    elif batch_size == 768:
        batch_size_768_configs.append(yaml_file_name.resolve())

    with open(yaml_file_name, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_0.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_1.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_2.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_3.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_4.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_5.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_6.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_7.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_8.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_9.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_10.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_11.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_12.yaml
../model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_13.yaml
../model_configs

In [7]:
manifest_dir = Path("train_config_manifests/")
manifest_dir.mkdir(parents=True, exist_ok=True)
out_name = manifest_dir / "single_task_ssl_hpara_search_bs_256.pkl"

config_dict_256 = {ix:config for ix,config in enumerate(batch_size_256_configs)} 

with open(out_name, 'wb') as f:
    pickle.dump(config_dict_256, f)
    

In [8]:
manifest_dir = Path("train_config_manifests/")
manifest_dir.mkdir(parents=True, exist_ok=True)
out_name = manifest_dir / "single_task_ssl_hpara_search_bs_768.pkl"

config_dict_768 = {ix:config for ix,config in enumerate(batch_size_768_configs)} 

with open(out_name, 'wb') as f:
    pickle.dump(config_dict_768, f)
    

## Set up Barlow Twins configs 

In [11]:
## Set hyperparameters to search over 

proj_dim = [np.power(2,pow) for pow in [9, 11, 13]]
print(proj_dim)
batch_size = [256, 768]
print(batch_size)
num_warmup_steps_or_ratio = [0.1] 
lr_scale = [0.6] # scaling factor for lr. lr will be lr_scale * batch_size / 256 
loss_lmbda = [0.005]


all_combos = list(product(*[proj_dim, proj_dim, batch_size]))
len(all_combos)


[512, 2048, 8192]
[256, 768]


18

In [13]:
## Set up MMCR configs 
config_dir = Path("../model_configs/barlow_search/")
config_dir.mkdir(parents=True, exist_ok=True)

base_config = yaml.load(open("../model_configs/pilot_ssl_audioset_resnet50_barlow.yaml"), Loader=yaml.FullLoader)
base_config

batch_size_256_configs = []
batch_size_768_configs = [] 

for ix, (proj_dim1, proj_dim2, batch_size) in enumerate(all_combos):
    yaml_file_name = config_dir / f"ssl_barlow_audioset_resnet50_hparam_set_{ix}.yaml"
    config = copy.deepcopy(base_config)
    # update model params 
    config['model']['arch_kwargs']['projector_dims'] = [int(proj_dim1), int(proj_dim2)]

    
    # update hyper parameters
    config['hparas']['global_batch_size'] = batch_size
    config['hparas']['ssl_loss_kwargs']['lmbda'] = loss_lmbda[0]
    config['hparas']['ssl_loss_kwargs']['out_dim'] = int(proj_dim2)

    # set standard params 
    config['lr'] = 0.6 # standard lr scale here 
    
    if batch_size == 256:
        batch_size_256_configs.append(yaml_file_name.resolve())
    elif batch_size == 768:
        batch_size_768_configs.append(yaml_file_name.resolve())

    with open(yaml_file_name, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

In [16]:
## Set up MMCR configs 
config_dir = Path("../model_configs/barlow_search/")
config_dir.mkdir(parents=True, exist_ok=True)

base_config = yaml.load(open("../model_configs/pilot_ssl_word_resnet50_barlow.yaml"), Loader=yaml.FullLoader)
base_config

batch_size_256_configs = []
batch_size_768_configs = [] 

for ix, (proj_dim1, proj_dim2, batch_size) in enumerate(all_combos):
    yaml_file_name = config_dir / f"ssl_barlow_word_resnet50_hparam_set_{ix}.yaml"
    config = copy.deepcopy(base_config)
    # update model params 
    config['model']['arch_kwargs']['projector_dims'] = [int(proj_dim1), int(proj_dim2)]

    
    # update hyper parameters
    config['hparas']['global_batch_size'] = batch_size
    config['hparas']['ssl_loss_kwargs']['lmbda'] = loss_lmbda[0]
    config['hparas']['ssl_loss_kwargs']['out_dim'] = int(proj_dim2)

    # set standard params 
    config['lr'] = 0.6 # standard lr scale here 
    
    if batch_size == 256:
        batch_size_256_configs.append(yaml_file_name.resolve())
    elif batch_size == 768:
        batch_size_768_configs.append(yaml_file_name.resolve())

    with open(yaml_file_name, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

In [18]:
manifest_dir = Path("../train_config_manifests/")
manifest_dir.mkdir(parents=True, exist_ok=True)
out_name = manifest_dir / "single_task_barlow_hpara_search_bs_256.pkl"

config_dict_256 = {ix:config for ix,config in enumerate(batch_size_256_configs)} 

with open(out_name, 'wb') as f:
    pickle.dump(config_dict_256, f)


out_name = manifest_dir / "single_task_barlow_hpara_search_bs_768.pkl"

config_dict_768 = {ix:config for ix,config in enumerate(batch_size_768_configs)} 

with open(out_name, 'wb') as f:
    pickle.dump(config_dict_768, f)
    
    

### Supervised Config

In [8]:
## Get list of configs to train 

config_list = sorted(list(Path("lightning_scripts/configs").glob("*lower*.yaml")))[1:] # [1:] since don't need original config
config_list

[PosixPath('lightning_scripts/configs/word_audioset_resnet50_lower_lr_lower_task_weight.yaml'),
 PosixPath('lightning_scripts/configs/word_audioset_resnet50_lower_lr_slower_schedule.yaml'),
 PosixPath('lightning_scripts/configs/word_audioset_resnet50_lower_lr_slower_schedule_lower_task_weight.yaml')]

In [11]:
config_dict = {ix:str(config) for ix,config in enumerate(config_list)}
config_dict

{0: 'lightning_scripts/configs/word_audioset_resnet50_lower_lr_lower_task_weight.yaml',
 1: 'lightning_scripts/configs/word_audioset_resnet50_lower_lr_slower_schedule.yaml',
 2: 'lightning_scripts/configs/word_audioset_resnet50_lower_lr_slower_schedule_lower_task_weight.yaml'}

In [13]:
manifest_dir = Path("train_config_manifests/")
manifest_dir.mkdir(parents=True, exist_ok=True)
out_name = manifest_dir / "word_audioset_supervised_hparam_search.pkl"

with open(out_name, 'wb') as f:
    pickle.dump(config_dict, f)